# AnimalAct: Fixed Splits and Main Evaluation

This notebook loads all 119 action classes, builds video-level multi-label targets, and reads the fixed main-evaluation class indices. A positive label means that the action occurs somewhere in the video; it does not encode temporal localization. The notebook defines evaluation utilities without training a model.

Run from the dataset root or the `demo` directory. All data paths are resolved relative to the dataset root.


In [ ]:
import json
from pathlib import Path
import numpy as np

ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("new_data/AnimalAct")]
             if (p / "manifest.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run from demo, AnimalAct, or repository root.")
manifest = json.loads((ROOT / "manifest.json").read_text())
vocab = json.loads((ROOT / "action_classes.json").read_text())
main = json.loads((ROOT / "main_evaluation_classes.json").read_text())
assert main["split_version"] == manifest["split_version"]
name_to_id = {c["name"]: c["class_id"] for c in vocab["classes"]}
assert sorted(name_to_id.values()) == list(range(119))
MAIN_CLASS_IDS = np.array(main["class_ids"], dtype=int)

def load_split(split):
    if split not in {"train", "valid", "test"}:
        raise ValueError("Use train, valid, or test")
    data = json.loads((ROOT / "splits" / f"{split}.json").read_text())
    assert data["split_version"] == manifest["split_version"]
    rows = data["samples"]
    targets = np.zeros((len(rows), 119), dtype=np.int64)
    for i, row in enumerate(rows):
        assert row["split"] == split
        assert (ROOT / row["video_path"]).is_file()
        ann = json.loads((ROOT / row["annotation_path"]).read_text())
        for animal in ann["annotations"]:
            for action in animal["actions"]:
                targets[i, name_to_id[action["action"]]] = 1
    return rows, targets

train_rows, y_train = load_split("train")
valid_rows, y_valid = load_split("valid")
test_rows, y_test = load_split("test")
for name, target in [("train", y_train), ("valid", y_valid), ("test", y_test)]:
    print(name, target.shape)
print("Main evaluation:", main["num_classes"], main["class_names"])


## Evaluation of Video-Level Predictions

A model must return 119 scores per test video. Columns correspond to global `class_id` values, and rows must follow `test_rows` exactly. After obtaining predictions, call `evaluate_main_map(y_test, test_scores)` to evaluate the fixed main classes.

The three-class image demo produces a different output and cannot be evaluated with this function. Average precision does not require a classification threshold; select thresholds for F1 on the validation set.


In [ ]:
from sklearn.metrics import average_precision_score

def evaluate_main_map(targets, scores):
    targets = np.asarray(targets)
    scores = np.asarray(scores)
    if targets.shape != (len(test_rows), 119) or scores.shape != targets.shape:
        raise ValueError("Expected test rows × 119, ordered exactly as test_rows and global class_id")
    if not np.array_equal(targets, y_test):
        raise ValueError("Targets must match the fixed test rows")
    if not np.isfinite(scores).all():
        raise ValueError("Scores must be finite")
    positive_counts = targets[:, MAIN_CLASS_IDS].sum(axis=0)
    if (positive_counts == 0).any():
        raise ValueError("A main-evaluation class has no positives; report the unsupported task explicitly")
    per_class = average_precision_score(targets[:, MAIN_CLASS_IDS], scores[:, MAIN_CLASS_IDS], average=None)
    return {"split_version": manifest["split_version"], "mAP": float(per_class.mean()),
            "per_class": [{"class_id": int(cid), "name": name, "AP": float(ap), "positive_videos": int(count)}
                          for cid, name, ap, count in zip(MAIN_CLASS_IDS, main["class_names"], per_class, positive_counts)]}
